# Degenerate FWM to coherent coupler transform check

In [37]:
from sympy import *
from time import time

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi, varsigma) = symbols(
    '''delta, nu, Aeff, chi, varsigma'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function
eta = IndexedBase('eta') 

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
alphabar = IndexedBase('alphabar', latex_name=r'\bar{\alpha}')
alphahat = IndexedBase('alphahat', latex_name=r'\hat{\alpha}')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_xy_addition = Eq(wpp(x + y, g2, g3),
    -(wpp(x, g2, g3) + wpp(y, g2, g3))/2 + 
    3*(wp(x, g2, g3) + wp(y, g2, g3))*(wpp(x, g2, g3) - wpp(y, g2, g3))/(2*(wp(x, g2, g3) - wp(y, g2, g3)))
    - (wpp(x, g2, g3) - wpp(y, g2, g3))**3/(4*(wp(x, g2, g3) - wp(y, g2, g3))**3)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)
pwp_sqrd = Eq(diff(wp(z,g2,g3),z)**2, 4*wp(z,g2,g3)**3 - g2*wp(z,g2,g3) - g3)
pwp_2nd = Eq(
    diff(wp(z,g2,g3),(z,2)),
    solve(Eq(diff(pwp_sqrd.lhs,z), diff(pwp_sqrd.rhs,z)), diff(wp(z,g2,g3),(z,2))
    )[0]
)



# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

quasi_period_sigma_b = Eq(
    sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3),
    (-1)**(m*n + m + n)*sigma(z, g2, g3)*
    exp(2*(m*omega[3] + n*omega[1] + z)*zeta(m*omega[3] + n*omega[1], g2, g3))
)

sigma_O2_abt_x = Eq(sigma(x + z, g2, g3), sigma(x, g2, g3) + z*sigma(x, g2, g3)*zeta(x, g2, g3) + O(z**2))


quasi_period_zw = Eq(
    zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 
    2*eta[1]*n + 2*eta[3]*m + zeta(z, g2, g3)
)
quasi_period_zw_b = Eq(
    zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 
    2*zeta(omega[1], g2, g3)*n + 2*zeta(omega[3], g2, g3)*m + zeta(z, g2, g3)
)
zw_omega_sum_0 = Eq(
    zeta(omega[1],g2,g3) + zeta(omega[3],g2,g3) - zeta(omega[1] + omega[3],g2,g3),
    0
)
zw_1_2_i_pi_omega = Eq(
    omega[1]*zeta(omega[3],g2,g3) - omega[3]*zeta(omega[1],g2,g3),
    I*pi/2
)
zw_m_n_omega = Eq(
    zeta(m*omega[3] + n*omega[1], g2, g3), 
    m*zeta(omega[3], g2, g3) + n*zeta(omega[1], g2, g3)
)


sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_xy_addition
pwp_sigma_dbl_ratio
pwp_sqrd
pwp_2nd

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

quasi_period_sigma_b
sigma_O2_abt_x

quasi_period_zw
quasi_period_zw_b
zw_omega_sum_0
zw_1_2_i_pi_omega
zw_m_n_omega

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 2*m*eta[3] + 2*n*eta[1] + zeta(z, g2, g3))

Eq(zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 2*m*zeta(omega[3], g2, g3) + 2*n*zeta(omega[1], g2, g3) + zeta(z, g2, g3))

Eq(-zeta(omega[1] + omega[3], g2, g3) + zeta(omega[1], g2, g3) + zeta(omega[3], g2, g3), 0)

Eq(-zeta(omega[1], g2, g3)*omega[3] + zeta(omega[3], g2, g3)*omega[1], I*pi/2)

Eq(zeta(m*omega[3] + n*omega[1], g2, g3), m*zeta(omega[3], g2, g3) + n*zeta(omega[1], g2, g3))

We remind ourselves of some parameter definitions:

In [3]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

## ----


gamma_j_prod_b_poly = Eq(
    Product(gamma[j] -  lamda[1], (j,1,2)), 
    (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4
)
C_gam_prod = Eq(Product(sqrt(gamma[j] - lamda[1]), (j,1,2)), C)
C_gam_prod_sqrd = Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)
C_gam_prod_sign = Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m * C)
chi_d5_C = Eq(chi, 16*(-1)**m*C/d[5])
C_d5_chi = Eq(C, solve(chi_d5_C,C)[0])

b_C_one_over_chi = Eq(
    (
        d5_dj.rhs
        .subs([_.args for _ in d_j_coeffs])
        .subs([_.args for _ in c_j_coeffs])
        .subs(b[0], solve(C_gam_prod_sign, b[0])[0])
        /(16*(-1)**m*C)
    )
    .expand(),
    (d5_dj.lhs/(16*(-1)**m*C)).subs([C_d5_chi.args])
)

d5_C_b_lam = Eq(
    d5_dj.lhs,
    d5_dj.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([(b[0], solve(C_gam_prod_sign, b[0])[0])])
    .expand()
    .collect(C, factor)
)



g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)



## ---

gam_sqrt_bj = C_gam_prod.doit().subs(gamma[2], -gamma[1])
gam_sqrt_bj = Eq(
    (2*gam_sqrt_bj.rhs/C_gam_prod_sign.rhs).subs((-1)**(-m), varsigma), 
    2*gam_sqrt_bj.lhs/C_gam_prod_sign.lhs
)

d5_bj_lam = Eq(
    d5_C_b_lam.lhs,
    d5_C_b_lam.rhs.subs(C_gam_prod_sign.rhs/2, C_gam_prod_sign.lhs/2)
    .expand().collect(lamda[1])
)

chi_bj_lam = chi_d5_C.subs([
    (C_gam_prod_sign.rhs, C_gam_prod_sign.lhs),
    d5_bj_lam.args
])


## --

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _
d4_lam_0
g2_dj
g3_dj

gamma_j_prod_b_poly
C_gam_prod
C_gam_prod_sqrd
C_gam_prod_sign
chi_d5_C
C_d5_chi
b_C_one_over_chi
d5_C_b_lam
gam_sqrt_bj
d5_bj_lam
chi_bj_lam

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(b[1]/4 + b[2]*lamda[1]/2 - lamda[1]/(2*(-1)**m*C), 1/chi)

Eq(d[5], 4*(-1)**m*C*(b[1] + 2*b[2]*lamda[1]) - 8*lamda[1])

Eq(varsigma, 2*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))

Eq(d[5], (4*b[0]*b[2] + 2*b[1]**2 - 8)*lamda[1] + 2*b[0]*b[1] + 6*b[1]*b[2]*lamda[1]**2 + 4*b[2]**2*lamda[1]**3)

Eq(chi, (8*b[0] + 8*b[1]*lamda[1] + 8*b[2]*lamda[1]**2)/((4*b[0]*b[2] + 2*b[1]**2 - 8)*lamda[1] + 2*b[0]*b[1] + 6*b[1]*b[2]*lamda[1]**2 + 4*b[2]**2*lamda[1]**3))

The starting point is the following single-parameter degenerate FWM system:

In [4]:
deg_dubar_dvbar = [
    Eq(
        Derivative(ubar(z, mu[1]), z),
        -varsigma*chi*vbar(z, mu[2])*vbar(z, mu[3])**2/4 + 
        chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - ubar(z, mu[1])/chi
    ),
    Eq(
        Derivative(ubar(z, mu[2]), z),
        -varsigma*chi*vbar(z, mu[1])*vbar(z, mu[3])**2/4 + 
        chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - ubar(z, mu[2])/chi
    ),
    Eq(
        Derivative(ubar(z, mu[3]), z),
        -varsigma*chi*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/2 + 
        chi*ubar(z, mu[3])**2*vbar(z, mu[3])/4 - ubar(z, mu[3])/chi
    ),
    Eq(
        Derivative(vbar(z, mu[1]), z),
        varsigma*chi*ubar(z, mu[2])*ubar(z, mu[3])**2/4 - 
        chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + vbar(z, mu[1])/chi
    ),
    Eq(
        Derivative(vbar(z, mu[2]), z),
        varsigma*chi*ubar(z, mu[1])*ubar(z, mu[3])**2/4 - 
        chi*ubar(z, mu[2])*vbar(z, mu[2])**2/2 + vbar(z, mu[2])/chi
    ),
    Eq(
        Derivative(vbar(z, mu[3]), z),
        varsigma*chi*ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])/2 - 
        chi*ubar(z, mu[3])*vbar(z, mu[3])**2/4 + vbar(z, mu[3])/chi
    )
]
deg_dubar_dvbar_subs = [_.args for _ in deg_dubar_dvbar]

H_deg_ubar_vbar = Eq(
    Hbar,
    -varsigma*chi*(
        ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + 
        vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2
    )/4 
    + chi*(
        2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 
        2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + 
        ubar(z, mu[3])**2*vbar(z, mu[3])**2
    )/8 
    - Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi
)

assert diff(H_deg_ubar_vbar.rhs.doit(),z).subs(deg_dubar_dvbar_subs).simplify() == 0, 'ham not conserved'


## Intermodal power

eps_vals = [
    Eq(epsilon[1], Rational(3/4)),
    Eq(epsilon[2], Rational(3/4)),
    Eq(epsilon[3], Rational(3/2))
]
eps_vals_subs = [_.args for _ in eps_vals]

rhobar_uvs = [
    Eq(
        ubar(z, mu[_j+1])*vbar(z, mu[_j+1]), gammabar[_j+1] - eps_vals_subs[_j][1]*rhobar(z)
    )#.subs(_eps_sols) 
    for _j in range(3)
]
rhobar_uv_subs = [_.args for _ in rhobar_uvs]

gammabar_sum = Eq(Sum(gammabar[j], (j,1,3)), 0)
rhobar_mean = Eq(
    -(sum(_.rhs for _ in rhobar_uvs)/3).subs(gammabar[3], solve(gammabar_sum.doit(), gammabar[3])[0]), 
    -sum(_.lhs for _ in rhobar_uvs)/3
)

_gambar_diffs = [
    ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]),
    ubar(z, mu[1])*vbar(z, mu[1])*2 - ubar(z, mu[3])*vbar(z, mu[3]),
    ubar(z, mu[2])*vbar(z, mu[2])*2 - ubar(z, mu[3])*vbar(z, mu[3])
]

gambar_pow_cons = [
    Eq(_, _.subs(rhobar_uv_subs)) for _ in _gambar_diffs
]

for _ in gambar_pow_cons:
    assert diff(_.lhs,z).doit().subs(deg_dubar_dvbar_subs).factor() == 0, 'intermodal power not conserved'



H_deg_ubar_vbar
print()
for _ in deg_dubar_dvbar:
    _
    
print()
gammabar_sum
rhobar_mean
for _ in rhobar_uvs:
    _
for _ in gambar_pow_cons:
    _

Eq(Hbar, -chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4 + chi*(2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + ubar(z, mu[3])**2*vbar(z, mu[3])**2)/8 - Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi)

Eq(Derivative(ubar(z, mu[1]), z), -chi*varsigma*vbar(z, mu[2])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - ubar(z, mu[1])/chi)

Eq(Derivative(ubar(z, mu[2]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - ubar(z, mu[2])/chi)

Eq(Derivative(ubar(z, mu[3]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/2 + chi*ubar(z, mu[3])**2*vbar(z, mu[3])/4 - ubar(z, mu[3])/chi)

Eq(Derivative(vbar(z, mu[1]), z), chi*varsigma*ubar(z, mu[2])*ubar(z, mu[3])**2/4 - chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + vbar(z, mu[1])/chi)

...

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -3*rhobar(z)/4 + gammabar[2])

Eq(ubar(z, mu[3])*vbar(z, mu[3]), -3*rhobar(z)/2 + gammabar[3])

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), gammabar[1] - gammabar[2])

Eq(2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[1] - gammabar[3])

Eq(2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[2] - gammabar[3])

Elsehwere we derived the below value for $\bar{H}$:

In [5]:
# Val if maps to coherent coupler
H_deg_ubar_vbar_val = Eq(Hbar, -b[2]*d[5]/8 - 2/chi + 64*(d[5] + 16*lamda[1])*lamda[1]/(chi**3*d[5]**2))
H_deg_ubar_vbar_val

Eq(Hbar, -b[2]*d[5]/8 - 2/chi + (64*d[5] + 1024*lamda[1])*lamda[1]/(chi**3*d[5]**2))

The target system we hope to transform the degenerate FWM system into is the following coherent coupler system:

In [6]:
Hhat_uv_b = Eq(
    -(gamma[1] - gamma[2])**2*b[2]/4 + b[0],
    uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + 
    Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2))
)

uvhat_rho = [
    Eq(uhat(z, mu[1])*vhat(z, mu[1]), gamma[1] - rho(z)),
    Eq(uhat(z, mu[2])*vhat(z, mu[2]), gamma[2] - rho(z))
]
uvhat_rho_subs = [_.args for _ in uvhat_rho]

uvhat_rho_ham_target = Eq(
    (Hhat_uv_b.lhs - Hhat_uv_b.rhs.doit())
    .subs(uvhat_rho_subs)
    .subs(gamma[2], - gamma[1])
    .expand().collect(rho(z), factor)
    , 
    0
)

Hhat_uv_b
uvhat_rho_ham_target
for _ in uvhat_rho:
    _

Eq(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0], uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2)))

Eq(rho(z)**2*b[2] + rho(z)*b[1] - uhat(z, mu[1])*uhat(z, mu[2]) - vhat(z, mu[1])*vhat(z, mu[2]) + b[0], 0)

Eq(uhat(z, mu[1])*vhat(z, mu[1]), -rho(z) + gamma[1])

Eq(uhat(z, mu[2])*vhat(z, mu[2]), -rho(z) + gamma[2])

In [7]:
gamma_lamda_bj_gam1 = Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)
gamma_lamda_bj_gam1

Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)

We believe, and want to prove, that the transform is given by:

In [8]:
u_v_hat_as_u_v_bar = [
    Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])*exp(z*theta)/vbar(z, mu[3])),
    Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])*exp(-z*theta)/vbar(z, mu[3])),
    Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])*exp(-z*theta)/ubar(z, mu[3])),
    Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])*exp(z*theta)/ubar(z, mu[3]))
]
u_v_hat_as_u_v_bar_subs = [_.args for _ in u_v_hat_as_u_v_bar]

for _ in u_v_hat_as_u_v_bar: 
    _

Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])*exp(z*theta)/vbar(z, mu[3]))

Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])*exp(-z*theta)/vbar(z, mu[3]))

Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])*exp(-z*theta)/ubar(z, mu[3]))

Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])*exp(z*theta)/ubar(z, mu[3]))

In [9]:
_uvhat_12_term = uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2])
uv_12_diff_hat = Eq(
    _uvhat_12_term, 
    _uvhat_12_term.subs(uvhat_rho_subs)
)

bars_to_3 = [
    Eq(ubar(z, mu[1])*vbar(z, mu[1]), solve(gambar_pow_cons[1],ubar(z, mu[1])*vbar(z, mu[1]))[0]),
    Eq(ubar(z, mu[2])*vbar(z, mu[2]), solve(gambar_pow_cons[2],ubar(z, mu[2])*vbar(z, mu[2]))[0])
]
bars_to_3_subs = [_.args for _ in bars_to_3]
uv_12_diff_hat_bar3 = uv_12_diff_hat.subs(u_v_hat_as_u_v_bar_subs).subs(bars_to_3_subs)
uv_12_diff_hat_bar3 = Eq( 
    uv_12_diff_hat_bar3.lhs.expand().collect(ubar(z,mu[3])*vbar(z,mu[3])),
    uv_12_diff_hat_bar3.rhs
)
gammabar_2_as_1 = Eq(
    gammabar[2], 
    solve(
        uv_12_diff_hat_bar3.subs([
        (gamma[2], -gamma[1]),
        (gammabar[3], -gammabar[1]-gammabar[2])
        ]).simplify(), 
        gammabar[2]
    )[0]
)
gamma_lam_as_gammabar = Eq(gamma[1]/lamda[1], solve(gammabar_2_as_1.subs(gamma[1], x*lamda[1]),x)[0])

uv_12_diff_hat
uv_12_diff_hat_bar3
gammabar_2_as_1
gamma_lam_as_gammabar

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

Eq((2*gamma[1]*gammabar[1] - gamma[1]*gammabar[3] - 2*gamma[2]*gammabar[2] + gamma[2]*gammabar[3] - 2*gammabar[1]*lamda[1] + 2*gammabar[2]*lamda[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + gamma[1] - gamma[2], gamma[1] - gamma[2])

Eq(gammabar[2], (-2*gamma[1] + lamda[1])*gammabar[1]/(2*gamma[1] + lamda[1]))

Eq(gamma[1]/lamda[1], (gammabar[1] - gammabar[2])/(2*(gammabar[1] + gammabar[2])))

In [10]:
_uvhat_12_gams_as_ubar3_term = (
    uhat(z, mu[2])*vhat(z, mu[2])*(gamma[1] - lamda[1]) -
    uhat(z, mu[1])*vhat(z, mu[1])*(gamma[2] - lamda[1])
)

uvhat_12_gams_as_ubar3 = Eq(
    _uvhat_12_gams_as_ubar3_term,
    _uvhat_12_gams_as_ubar3_term
    .subs(u_v_hat_as_u_v_bar_subs)
    .factor()
    .subs([gambar_pow_cons[0].args])
)
delta_gam_gambar = Eq(delta, fraction(uvhat_12_gams_as_ubar3.rhs)[0])

ubar3_as_uvhat_12_gams = Eq(
    (delta_gam_gambar.rhs/uvhat_12_gams_as_ubar3.rhs),
    (delta_gam_gambar.lhs/uvhat_12_gams_as_ubar3.lhs)
)

uvhat_12_gams_as_ubar3
delta_gam_gambar
ubar3_as_uvhat_12_gams

Eq((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]), -2*(gamma[1] - lamda[1])*(gamma[2] - lamda[1])*(gammabar[1] - gammabar[2])/(ubar(z, mu[3])*vbar(z, mu[3])))

Eq(delta, -2*(gamma[1] - lamda[1])*(gamma[2] - lamda[1])*(gammabar[1] - gammabar[2]))

Eq(ubar(z, mu[3])*vbar(z, mu[3]), delta/((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1])))

In [11]:
ubar_vbar_from_u_v_hat = [
    Eq(ubar(z,mu[1]), solve(u_v_hat_as_u_v_bar[0], ubar(z,mu[1]))[0]),
    Eq(ubar(z,mu[2]), solve(u_v_hat_as_u_v_bar[1], ubar(z,mu[2]))[0]),
    Eq(vbar(z,mu[1]), solve(u_v_hat_as_u_v_bar[2], vbar(z,mu[1]))[0]),
    Eq(vbar(z,mu[2]), solve(u_v_hat_as_u_v_bar[3], vbar(z,mu[2]))[0])
]
ubar_vbar_from_u_v_hat_subs = [_.args for _ in ubar_vbar_from_u_v_hat]

for _ in u_v_hat_as_u_v_bar:
    Eq(
        diff(_.lhs,z), 
        diff(_.rhs,z)
        .subs(deg_dubar_dvbar_subs)
        .subs(ubar_vbar_from_u_v_hat_subs)
        .expand().collect([varsigma, ubar(z, mu[3])*vbar(z, mu[3])], factor)
        .subs([ubar3_as_uvhat_12_gams.args])
    )

Eq(Derivative(uhat(z, mu[1]), z), -chi*delta*varsigma*(uhat(z, mu[1])**2*uhat(z, mu[2]) + vhat(z, mu[2])*gamma[1] - vhat(z, mu[2])*lamda[1])/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) + chi*delta*(uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])*uhat(z, mu[1])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[1] - 4*lamda[1])) + (chi*theta - 2)*uhat(z, mu[1])/chi)

Eq(Derivative(uhat(z, mu[2]), z), -chi*delta*varsigma*(uhat(z, mu[1])*uhat(z, mu[2])**2 + vhat(z, mu[1])*gamma[2] - vhat(z, mu[1])*lamda[1])/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) + chi*delta*(uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])*uhat(z, mu[2])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[2] - 4*lamda[1])) - (chi*theta + 2)*uhat(z, mu[2])/chi)

Eq(Derivative(vhat(z, mu[1]), z), chi*delta*varsigma*(uhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*lamda[1] + vhat(z, mu[1])**2*vhat(z, mu[2]))/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) - chi*delta*(uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])*vhat(z, mu[1])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[1] - 4*lamda[1])) - (chi*theta - 2)*vhat(z, mu[1])/chi)

Eq(Derivative(vhat(z, mu[2]), z), chi*delta*varsigma*(uhat(z, mu[1])*gamma[2] - uhat(z, mu[1])*lamda[1] + vhat(z, mu[1])*vhat(z, mu[2])**2)/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) - chi*delta*(uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])*vhat(z, mu[2])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[2] - 4*lamda[1])) + (chi*theta + 2)*vhat(z, mu[2])/chi)

We will make two choices. First, we fix a value for $\bar{\gamma}_1 - \bar{\gamma}_1$, then we fix a value for $\chi$. We note that our choice for $\bar{\gamma}_1 - \bar{\gamma}_1$ is not inconsistent with the map:

In [12]:
gambar_dif_choice = Eq(
    gammabar[1] - gammabar[2], 
    gamma[1]*d[5]/(4*(gamma[1]**2 - lamda[1]**2))
)
gammabar_gamma_subs = solve([gamma_lam_as_gammabar, gambar_dif_choice], [gammabar[1], gammabar[2]])

# Define d6
d6_d5 = Eq(d[6], d[5]/(8*(gamma[1]**2 - lamda[1]**2)))
d6_b = Eq(
    d[6],
    -(
        2*(2*b[0]*b[2] + b[1]**2 - 4)*lamda[1] + 2*b[0]*b[1] + 
        6*b[1]*b[2]*lamda[1]**2 + 4*b[2]**2*lamda[1]**3
    )/(2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2)
)

# These are choices we know work but they came from the wp solutions so here we say that we choose them
# (note consistent with derived expression for gamma[1]/lambda[1])
gam1_d6 = Eq(gamma[1], (gammabar[1] - gammabar[2])/(2*d[6]))
lam1_d6 = Eq(lamda[1], (gammabar[1] + gammabar[2])/d[6])

gammabar_1_2_as_gamma = [ Eq(_k, _v) for _k, _v in gammabar_gamma_subs.items()]

d6_d5
d6_b
gam1_d6
lam1_d6
gambar_dif_choice

for _ in gammabar_1_2_as_gamma:
    _

Eq(d[6], d[5]/(8*gamma[1]**2 - 8*lamda[1]**2))

Eq(d[6], (-(4*b[0]*b[2] + 2*b[1]**2 - 8)*lamda[1] - 2*b[0]*b[1] - 6*b[1]*b[2]*lamda[1]**2 - 4*b[2]**2*lamda[1]**3)/(2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2))

Eq(gamma[1], (gammabar[1] - gammabar[2])/(2*d[6]))

Eq(lamda[1], (gammabar[1] + gammabar[2])/d[6])

Eq(gammabar[1] - gammabar[2], d[5]*gamma[1]/(4*gamma[1]**2 - 4*lamda[1]**2))

Eq(gammabar[1], (2*gamma[1] + lamda[1])*d[5]/(16*(gamma[1]**2 - lamda[1]**2)))

Eq(gammabar[2], (-2*gamma[1] + lamda[1])*d[5]/(16*(gamma[1]**2 - lamda[1]**2)))

In [13]:
delta_d5 = delta_gam_gambar.subs([gambar_dif_choice.args, (gamma[2], -gamma[1])]).simplify()

chi_delta = chi_d5_C.subs([
    (C_gam_prod.rhs, C_gam_prod.lhs), 
    ((-1)**m, varsigma), 
    (d[5], solve(delta_d5, d[5])[0])
]).doit()

chi_d5_sqrt_gam = chi_delta.subs([delta_d5.args])

delta_d5
chi_delta
chi_d5_sqrt_gam

Eq(delta, d[5]*gamma[1]/2)

Eq(chi, 8*varsigma*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*gamma[1]/delta)

Eq(chi, 16*varsigma*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

We confirm that the DFWM Hamiltonian then transforms into the target Hamiltonian for the coherent coupler. We have assumed the value for $\bar{H}$ derived elsewhere:

In [14]:
H_deg_to_rho_hat = Eq(
    (
        (
            H_deg_ubar_vbar.lhs
            .subs([H_deg_ubar_vbar_val.args])
            - H_deg_ubar_vbar.rhs
            .doit()
            .subs(ubar_vbar_from_u_v_hat_subs)
        )/(ubar(z, mu[3])*vbar(z, mu[3]))**2/(2/d[5])
    )
    .expand().collect(ubar(z, mu[3])*vbar(z, mu[3]), factor)
    .subs([
        ubar3_as_uvhat_12_gams.args, 
        chi_delta.args,
        gam_sqrt_bj.args,
        delta_d5.args
    ])
    .subs(uvhat_rho_subs)
    .subs([
        (gamma[2], -gamma[1])
    ])
    .expand().collect([rho(z), uhat(z, mu[1]), vhat(z, mu[1])], simplify)
    .subs([gamma_lamda_bj_gam1.args])
    .collect([rho(z), uhat(z, mu[1]), vhat(z, mu[1])], simplify)
    .subs([d5_bj_lam.args])
    .collect([rho(z), uhat(z, mu[1]), vhat(z, mu[1])], simplify)
    ,
    0
)

assert uvhat_rho_ham_target.subs(b[0], solve(H_deg_to_rho_hat,b[0])[0]), 'target ham not obtained'

H_deg_to_rho_hat

Eq(-rho(z)**2*b[2] - rho(z)*b[1] + uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) - b[0], 0)

## Solving the DFWM system

In [15]:
H_deg_ubar_vbar
print()
for _ in deg_dubar_dvbar:
    _

print()
gammabar_sum
rhobar_mean
for _ in rhobar_uvs:
    _
for _ in gambar_pow_cons:
    _

Eq(Hbar, -chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4 + chi*(2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + ubar(z, mu[3])**2*vbar(z, mu[3])**2)/8 - Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi)

Eq(Derivative(ubar(z, mu[1]), z), -chi*varsigma*vbar(z, mu[2])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - ubar(z, mu[1])/chi)

Eq(Derivative(ubar(z, mu[2]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - ubar(z, mu[2])/chi)

Eq(Derivative(ubar(z, mu[3]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/2 + chi*ubar(z, mu[3])**2*vbar(z, mu[3])/4 - ubar(z, mu[3])/chi)

Eq(Derivative(vbar(z, mu[1]), z), chi*varsigma*ubar(z, mu[2])*ubar(z, mu[3])**2/4 - chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + vbar(z, mu[1])/chi)

...

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -3*rhobar(z)/4 + gammabar[2])

Eq(ubar(z, mu[3])*vbar(z, mu[3]), -3*rhobar(z)/2 + gammabar[3])

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), gammabar[1] - gammabar[2])

Eq(2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[1] - gammabar[3])

Eq(2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[2] - gammabar[3])

In [16]:
wm_deg_ubar_vbar = Eq(
    H_deg_ubar_vbar.rhs - H_deg_ubar_vbar.rhs.subs(varsigma,0),
    H_deg_ubar_vbar.lhs - H_deg_ubar_vbar.rhs.subs(varsigma,0)
)

duvbar1 = Eq(
    Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z),
    Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z)
    .doit()
    .subs(deg_dubar_dvbar_subs)
    .factor()
)

drhobar_sqrd = Eq(
    (duvbar1.lhs**2)
    .subs(rhobar_uv_subs)
    .doit()*Rational(16,9),
    (((duvbar1.rhs**2) - wm_deg_ubar_vbar.lhs**2 + wm_deg_ubar_vbar.rhs**2)*Rational(16,9))
    .doit().expand()
    .subs(rhobar_uv_subs)
    .expand().collect(rhobar(z), simplify)
    .subs([
        (varsigma**2, 1),
        (gammabar[3], solve(gammabar_sum.doit(), gammabar[3])[0])
    ])
    .collect(rhobar(z), simplify)
)

duvbar1
wm_deg_ubar_vbar
drhobar_sqrd

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4)

Eq(-chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4, Hbar - chi*(2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + ubar(z, mu[3])**2*vbar(z, mu[3])**2)/8 + Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi)

Eq(Derivative(rhobar(z), z)**2, 16*Hbar**2/9 - 4*Hbar*chi*gammabar[1]**2/3 - 8*Hbar*chi*gammabar[1]*gammabar[2]/9 - 4*Hbar*chi*gammabar[2]**2/3 + chi**2*gammabar[1]**4/4 - chi**2*gammabar[1]**3*gammabar[2]/9 - 5*chi**2*gammabar[1]**2*gammabar[2]**2/18 - chi**2*gammabar[1]*gammabar[2]**3/9 + chi**2*gammabar[2]**4/4 + 6*rhobar(z)**3 + (-32*Hbar + chi*(chi**2*gammabar[1]**3 - chi**2*gammabar[1]**2*gammabar[2] - chi**2*gammabar[1]*gammabar[2]**2 + chi**2*gammabar[2]**3 + 12*gammabar[1]**2 + 8*gammabar[1]*gammabar[2] + 12*gammabar[2]**2))*rhobar(z)/(3*chi) + (chi**3*(-4*Hbar + 3*chi*gammabar[1]**2 + 2*chi*gammabar[1]*gammabar[2] + 3*chi*gammabar[2]**2) + 32)*rhobar(z)**2/(2*chi**2))

In [17]:
w_rhobar = Eq(rhobar(z), c[1]*w(z) + c[0])

_c1_sub_2_3 = Eq(c[1], Rational(2,3))
# elliminate the quadratic term by solving for c[0]
dw_sqrd = Eq(
    drhobar_sqrd.lhs.subs([w_rhobar.args]).doit()/c[1]**2,
    (drhobar_sqrd.rhs.subs([w_rhobar.args])/c[1]**2)
    .expand().collect(w(z), factor)
    .subs([_c1_sub_2_3.args])
)
_c0_sol = Eq(c[0], solve(dw_sqrd.rhs.coeff(w(z),2), c[0])[0])
dw_sqrd = Eq(
    dw_sqrd.lhs, 
    sum(
        _.collect([Hbar, chi], factor)
        for _ in 
        dw_sqrd.rhs.subs([_c0_sol.args])
        .expand().collect(w(z), factor)
        .args
    )
)

params_to_bj_lam = [
    gamma_lamda_bj_gam1.args,
    d5_bj_lam.args,
    d6_b.args,
    chi_bj_lam.args
]

g2_bj_lam = Eq(
    g2_dj.lhs,
    g2_dj.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([_.args for _ in d_j_coeffs])
    .doit()
    .subs(gamma[2], -gamma[1])
    .subs(params_to_bj_lam)
    .expand().collect(lamda[1], factor)
)
g3_bj_lam = Eq(
    g3_dj.lhs,
    g3_dj.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([_.args for _ in d_j_coeffs])
    .doit()
    .subs(gamma[2], -gamma[1])
    .subs(params_to_bj_lam)
    .expand().collect(lamda[1], factor)
)

# check we obtain g2, g3 based on their known values in dj
_gam_lam_subs = solve([gam1_d6, lam1_d6], [gammabar[1], gammabar[2]])

_Hbar_chi_g2_sub = Eq(
    dw_sqrd.rhs.coeff(w(z),1),
    dw_sqrd.rhs.coeff(w(z),1)
    .subs([H_deg_ubar_vbar_val.args])
    .subs(_gam_lam_subs)
    .expand().collect(gamma[1])
    .subs(params_to_bj_lam)
    .simplify()
    .expand().collect(lamda[1], factor)
    .subs(g2_bj_lam.rhs, g2_bj_lam.lhs)
)


_Hbar_chi_g3_sub = Eq(
    dw_sqrd.rhs.coeff(w(z),0),
    dw_sqrd.rhs.coeff(w(z),0)
    .subs([H_deg_ubar_vbar_val.args])
    .subs(_gam_lam_subs)
    .expand().collect(gamma[1])
    .subs(params_to_bj_lam)
    .simplify()
    .expand().collect(lamda[1], factor)
    .subs(g3_bj_lam.rhs, g3_bj_lam.lhs)
)


dw_sqrd_g2_g3 = Eq(
    dw_sqrd.lhs,
    (
        dw_sqrd.rhs
        - _Hbar_chi_g2_sub.lhs*w(z) + _Hbar_chi_g2_sub.rhs*w(z)
        - _Hbar_chi_g3_sub.lhs + _Hbar_chi_g3_sub.rhs
    ).collect(w(z), factor)
)

# express c[0] in b_j

_c0_sol_b = Eq(
    _c0_sol.lhs,
    _c0_sol.rhs
    .subs([H_deg_ubar_vbar_val.args])
    .subs(_gam_lam_subs)
    .subs(params_to_bj_lam)
    .factor()
    .subs([gamma_lamda_bj_gam1.args])
    .factor()
    .subs(b[0], x - b[1]*lamda[1] - b[2]*lamda[1]**2)
    .expand().collect(x, factor)
    .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
)

# define final rhobar as w
w_rhobar_b = Eq(w_rhobar.lhs, w_rhobar.rhs.subs([_c1_sub_2_3.args, _c0_sol_b.args]))
wz_wp = Eq(w(z), wp(z-z0, g2, g3))

# w_rhobar
# dw_sqrd
dw_sqrd_g2_g3
wz_wp
w_rhobar_b

Eq(Derivative(w(z), z)**2, -g2*w(z) - g3 + 4*w(z)**3)

Eq(w(z), wp(z - z0, g2, g3))

Eq(rhobar(z), 2*(b[1] + 2*b[2]*lamda[1])*lamda[1]/(3*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)) - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*b[2]/9 - (b[1] + 2*b[2]*lamda[1] - 2)*(b[1] + 2*b[2]*lamda[1] + 2)/18 + 2*w(z)/3 - 8*lamda[1]**2/(3*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2))

We recall from our solution to the coherent coupler that points $\mu_j, z_1$ are defined in terms of Weierstrass elliptic functions and parameters as follows:

In [18]:
B_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(gamma[j] - lamda[1])*b[2]
)
C_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)/(gamma[j] - lamda[1])
)
D_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)*wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2,
    (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*b[2]
)


wp_z1_lam_uv_j = Eq(
    wpp(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*b[2]),
    -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2
)
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)
wppz1_djs = Eq( 4*wpp(z1, g2, g3)/b[2], -d[5])

wp_z0_mu_j_d = Eq(
    wp(-z0 + mu[j], g2, g3),
    d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - 
    (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1]))
)
wpp_z0_mu_j_d = Eq(
    wpp(-z0 + mu[j], g2, g3),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)
    *(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2)
)

z1_wp = Eq(wp(z1, g2, g3), (d[2] + 3*d[3]*lamda[1] + 6*d[4]*lamda[1]**2)/12)


B_wpz1_z0_muj_to_d_lam1_gamj
C_wpz1_z0_muj_to_d_lam1_gamj
D_wpz1_z0_muj_to_d_lam1_gamj
wp_z1_lam_uv_j
wppz1_djs
wp_z0_mu_j_d
wpp_z0_mu_j_d
d4_lam_0
d5_dj
z1_wp

Eq(\wp'(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-gamma[j] + lamda[1])*b[2])

Eq(\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)/(gamma[j] - lamda[1]))

Eq(\wp'(z1, g2, g3)*\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2, (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*b[2])

Eq(\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*b[2]), -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2)

Eq(4*\wp'(z1, g2, g3)/b[2], -d[5])

Eq(wp(-z0 + mu[j], g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*gamma[j] - 4*lamda[1]))

Eq(\wp'(-z0 + mu[j], g2, g3), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)*(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2))

Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)

Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

Eq(wp(z1, g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2)

Thus we confirm that we obtain the expected solutions for modal powers:

In [19]:
rhobar_uvs_w_b = [ 
    _.subs([w_rhobar_b.args])
    .subs(gammabar[3], -gammabar[1] - gammabar[2])
    .subs(gammabar_gamma_subs)
    .subs(params_to_bj_lam)
    for _ in rhobar_uvs
]

wp_mu1_bj = Eq(
    wp_z0_mu_j_d.lhs.subs(j,1),
    wp_z0_mu_j_d.rhs.subs(j,1)
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([_.args for _ in d_j_coeffs])
    .doit()
    .subs(gamma[2], -gamma[1])
)
wp_mu2_bj = Eq(
    wp_z0_mu_j_d.lhs.subs(j,2),
    wp_z0_mu_j_d.rhs.subs(j,2)
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([_.args for _ in d_j_coeffs])
    .doit()
    .subs(gamma[2], -gamma[1])
)
wp_z1_bj = Eq(
    z1_wp.lhs,
    z1_wp.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([_.args for _ in d_j_coeffs])
    .doit()
    .subs(gamma[2], -gamma[1])
)

uvbar_wps = [
    Eq(
        rhobar_uvs_w_b[0].lhs,
        (rhobar_uvs_w_b[0].rhs - wp_mu1_bj.rhs/2)
        .expand().collect(w(z), factor)
        .subs(params_to_bj_lam)
        .collect(w(z), factor)
        .subs([wz_wp.args])
        + wp_mu1_bj.lhs/2
    ),
    Eq(
        rhobar_uvs_w_b[1].lhs,
        (rhobar_uvs_w_b[1].rhs - wp_mu2_bj.rhs/2)
        .expand().collect(w(z), factor)
        .subs(params_to_bj_lam)
        .collect(w(z), factor)
        .subs([wz_wp.args])
        + wp_mu2_bj.lhs/2
    ),
    Eq(
        rhobar_uvs_w_b[2].lhs,
        (rhobar_uvs_w_b[2].rhs - wp_z1_bj.rhs)
        .expand().collect(w(z), factor)
        .subs(params_to_bj_lam)
        .collect(w(z), factor)
        .subs([wz_wp.args])
        + wp_z1_bj.lhs
    )
]
uvbar_wp_subs = [_.args for _ in uvbar_wps]

for _ in uvbar_wps:
    _

Eq(ubar(z, mu[1])*vbar(z, mu[1]), -wp(z - z0, g2, g3)/2 + wp(-z0 + mu[1], g2, g3)/2)

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -wp(z - z0, g2, g3)/2 + wp(-z0 + mu[2], g2, g3)/2)

Eq(ubar(z, mu[3])*vbar(z, mu[3]), wp(z1, g2, g3) - wp(z - z0, g2, g3))

In [20]:
gammabars_as_wp = [_.subs(uvbar_wp_subs) for _ in gambar_pow_cons]

_gambar12_sols_dict = solve(
        [_.subs(gammabar[3], -gammabar[1] - gammabar[2]) for _ in gammabars_as_wp], 
        [gammabar[_j+1] for _j in range(3)]
    )
gammabars_as_wp_sols = [
    Eq(_k, _v) 
    for _k, _v in _gambar12_sols_dict.items()
]
gammabars_as_wp_sols.append(Eq(gammabar[3], ( -gammabar[1] - gammabar[2]).subs(_gambar12_sols_dict)))

for _ in gammabars_as_wp_sols:
    _

Eq(gammabar[1], -wp(z1, g2, g3)/4 + 3*wp(-z0 + mu[1], g2, g3)/8 - wp(-z0 + mu[2], g2, g3)/8)

Eq(gammabar[2], -wp(z1, g2, g3)/4 - wp(-z0 + mu[1], g2, g3)/8 + 3*wp(-z0 + mu[2], g2, g3)/8)

Eq(gammabar[3], wp(z1, g2, g3)/2 - wp(-z0 + mu[1], g2, g3)/4 - wp(-z0 + mu[2], g2, g3)/4)

We now derive expressions for the logarithmic derivatives of modes:

In [21]:
# Express the log diff of u_j in terms of modal power w_j

Hbar_diff_u = Eq(
    (wm_deg_ubar_vbar.lhs + duvbar1.rhs)
    .factor()/2,
    (wm_deg_ubar_vbar.rhs + duvbar1.lhs)/2
)
Hbar_diff_v = Eq(
    (wm_deg_ubar_vbar.lhs - duvbar1.rhs)
    .factor()/2,
    (wm_deg_ubar_vbar.rhs - duvbar1.lhs)/2
)

Hbar_diff_u1 = Eq(
    Hbar_diff_u.lhs/(ubar(z, mu[1])*vbar(z, mu[1])), 
    (Hbar_diff_u.rhs/(ubar(z, mu[1])*vbar(z, mu[1])))
    .subs(ubar(z, mu[1])*vbar(z, mu[1]), w(z,1))
    .doit()
    .subs([
        (ubar(z, mu[2])*vbar(z, mu[2]), solve(gambar_pow_cons[0], ubar(z, mu[2])*vbar(z, mu[2]))[0]),
        (ubar(z, mu[3])*vbar(z, mu[3]), solve(gambar_pow_cons[1], ubar(z, mu[3])*vbar(z, mu[3]))[0]),
        (gammabar[3], -gammabar[1] - gammabar[2])
    ])
    .subs(ubar(z, mu[1])*vbar(z, mu[1]), w(z,1))
    .expand().collect(w(z,1), factor)
)
Hbar_diff_u2 = Eq(
    Hbar_diff_u.lhs/(ubar(z, mu[2])*vbar(z, mu[2])), 
    (
        Hbar_diff_u.rhs
        .subs(
            ubar(z, mu[1])*vbar(z, mu[1]), 
            solve(gambar_pow_cons[0], ubar(z, mu[1])*vbar(z, mu[1])
                 )[0]
        )/(ubar(z, mu[2])*vbar(z, mu[2]))
    )
    .subs(ubar(z, mu[2])*vbar(z, mu[2]), w(z,2))
    .doit()
    .subs([
        (ubar(z, mu[1])*vbar(z, mu[1]), solve(gambar_pow_cons[0], ubar(z, mu[1])*vbar(z, mu[1]))[0]),
        (ubar(z, mu[3])*vbar(z, mu[3]), solve(gambar_pow_cons[2], ubar(z, mu[3])*vbar(z, mu[3]))[0]),
        (gammabar[3], -gammabar[1] - gammabar[2])
    ])
    .subs(ubar(z, mu[2])*vbar(z, mu[2]), w(z,2))
    .expand().collect(w(z,2), factor)
)
Hbar_diff_u3 = Eq(
    2*Hbar_diff_u.lhs/(ubar(z, mu[3])*vbar(z, mu[3])), 
    (
        2*Hbar_diff_u.rhs
        .subs(
            ubar(z, mu[1])*vbar(z, mu[1]), 
            solve(gambar_pow_cons[1], ubar(z, mu[1])*vbar(z, mu[1])
                 )[0]
        )/(ubar(z, mu[3])*vbar(z, mu[3]))
    )
    .subs(ubar(z, mu[3])*vbar(z, mu[3]), w(z,3))
    .doit()
    .subs([
        (ubar(z, mu[1])*vbar(z, mu[1]), solve(gambar_pow_cons[0], ubar(z, mu[1])*vbar(z, mu[1]))[0]),
        (ubar(z, mu[2])*vbar(z, mu[2]), solve(gambar_pow_cons[2], ubar(z, mu[2])*vbar(z, mu[2]))[0]),
        (gammabar[2], -gammabar[1] - gammabar[3])
    ])
    .subs(ubar(z, mu[3])*vbar(z, mu[3]), w(z, 3))
    .expand().collect(w(z,3), factor)
)

dlog_ubars = [
    Eq(
        deg_dubar_dvbar[0].lhs/ubar(z, mu[1]),
        (deg_dubar_dvbar[0].rhs/ubar(z, mu[1]))
        .expand()
        .subs([
            (ubar(z, mu[1])*vbar(z, mu[1]), w(z,1)),
            Hbar_diff_u1.args
        ])
        .expand().collect(w(z,1), factor)
    ),
    Eq(
        deg_dubar_dvbar[1].lhs/ubar(z, mu[2]),
        (deg_dubar_dvbar[1].rhs/ubar(z, mu[2]))
        .expand()
        .subs([
            (ubar(z, mu[2])*vbar(z, mu[2]), w(z,2)),
            Hbar_diff_u2.args
        ])
        .expand().collect(w(z,2), factor)
    ),
    Eq(
        deg_dubar_dvbar[2].lhs/ubar(z, mu[3]),
        (deg_dubar_dvbar[2].rhs/ubar(z, mu[3]))
        .expand()
        .subs([
            (ubar(z, mu[3])*vbar(z, mu[3]), w(z,3)),
            Hbar_diff_u3.args
        ])
        .expand().collect(w(z,3), factor)
    )
]

for _ in dlog_ubars:
    _

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), Derivative(w(z, 1), z)/(2*w(z, 1)) + (chi**2*gammabar[1] + 1)/chi + (8*Hbar*chi - 11*chi**2*gammabar[1]**2 - 2*chi**2*gammabar[1]*gammabar[2] - 3*chi**2*gammabar[2]**2 - 32*gammabar[1])/(16*chi*w(z, 1)))

Eq(Derivative(ubar(z, mu[2]), z)/ubar(z, mu[2]), Derivative(w(z, 2), z)/(2*w(z, 2)) + (chi**2*gammabar[2] + 1)/chi + (8*Hbar*chi - 3*chi**2*gammabar[1]**2 - 2*chi**2*gammabar[1]*gammabar[2] - 11*chi**2*gammabar[2]**2 - 32*gammabar[2])/(16*chi*w(z, 2)))

Eq(Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]), Derivative(w(z, 3), z)/(2*w(z, 3)) + (chi**2*gammabar[3] + 2)/(2*chi) + (8*Hbar*chi - 4*chi**2*gammabar[1]**2 - 4*chi**2*gammabar[1]*gammabar[3] - 5*chi**2*gammabar[3]**2 - 16*gammabar[3])/(8*chi*w(z, 3)))

In [22]:
# Express the constant part above w_j in terms of b_j, gamma_j, lamda_1 to compare with wp' values

gam1_rec_1 = Eq(
    -(gamma_lamda_bj_gam1.lhs - lamda[1]**2).factor()/
    (gamma_lamda_bj_gam1.rhs - lamda[1]**2)/
    (gamma[1] - lamda[1])/4,
    -1/(gamma[1] - lamda[1])/4
)

gam1_rec_2 = Eq(
    gam1_rec_1.lhs*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2),
    gam1_rec_1.rhs*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
)

H_b_s=[
    Eq(
        dlog_ubars[0].rhs.coeff(chi*w(z,1), -1),
        dlog_ubars[0].rhs.coeff(chi*w(z,1), -1)
        .subs([H_deg_ubar_vbar_val.args])
        .subs(gammabar[3], -gammabar[1] - gammabar[2])
        .subs(_gam_lam_subs)
        .subs(params_to_bj_lam)
        .factor()
        .subs([gamma_lamda_bj_gam1.args])
        .subs(b[0], x- b[1]*lamda[1]- b[2]*lamda[1]**2)
        .expand().collect(x, factor)
        .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
        .subs([
            gam1_rec_1.args,
            gam1_rec_2.args
        ])
    ),
    Eq(
        dlog_ubars[1].rhs.coeff(chi*w(z,2), -1),
        dlog_ubars[1].rhs.coeff(chi*w(z,2), -1)
        .subs([H_deg_ubar_vbar_val.args])
        .subs(gammabar[3], -gammabar[1] - gammabar[2])
        .subs(_gam_lam_subs)
        .subs(params_to_bj_lam)
        .factor()
        .subs([gamma_lamda_bj_gam1.args])
        .subs(b[0], x- b[1]*lamda[1]- b[2]*lamda[1]**2)
        .expand().collect(x, factor)
        .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
        .subs([
            (1/gam1_rec_1.rhs/4, 1/gam1_rec_1.lhs/4)
        ])
    ),
    Eq(
        dlog_ubars[2].rhs.coeff(chi*w(z,3), -1),
        dlog_ubars[2].rhs.coeff(chi*w(z,3), -1)
        .subs([H_deg_ubar_vbar_val.args])
        .subs(gammabar[3], -gammabar[1] - gammabar[2])
        .subs(_gam_lam_subs)
        .subs(params_to_bj_lam)
        .factor()
        .subs([gamma_lamda_bj_gam1.args])
        .subs(b[0], x- b[1]*lamda[1]- b[2]*lamda[1]**2)
        .expand().collect(x, simplify)
        .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
    )
]


# Express the wp' values in terms of b_j, gamma_j, lamda_1

_glj = gamma_lamda_bj_gam1.subs(gamma[1], gamma[j])
glj_b_sub = Eq(
    -4*(_glj.rhs - lamda[1]**2).factor(),
    -4*(_glj.lhs - lamda[1]**2).factor()
)
_gl_ratio = (gamma[j] + lamda[1])/(gamma[j] - lamda[1])
_gl_apart_sub = Eq(2*_gl_ratio, apart(2*_gl_ratio, gamma[j]))

chi_wp_muj = Eq(
    wpp_z0_mu_j_d.lhs*chi/4,
    (wpp_z0_mu_j_d.rhs*chi/4)
    .subs(params_to_bj_lam)
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([_.args for _ in d_j_coeffs])
    .doit()
    .subs(gamma[2], -gamma[1])
    .factor()
    .subs([gamma_lamda_bj_gam1.args])
    .subs(b[0], x- b[1]*lamda[1]- b[2]*lamda[1]**2)
    .expand().collect([b[1], b[2], x], factor)
    .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
    .subs(gamma[j], y+lamda[1])
    .expand().collect(y, factor)
    .subs([(y, gamma[j] - lamda[1]), glj_b_sub.args, _gl_apart_sub.args])
)
chi_wp_z1 = Eq(
    wpp(z1, g2, g3)*chi/2, 
    (solve(wppz1_djs, wpp(z1, g2, g3))[0]*chi/2)
    .subs(params_to_bj_lam)
    .factor()
)

# Join the two expressions to enable subbing in wp' values

H_b_s_wp = [
    Eq(
        16*H_b_s[0].lhs, 
        16*((H_b_s[0].rhs - chi_wp_muj.rhs.subs(j,1)).simplify() 
        + chi_wp_muj.lhs.subs(j,1))
    ),
    Eq(
        16*H_b_s[1].lhs, 
        16*((H_b_s[1].rhs - chi_wp_muj.rhs.subs(j,2).subs(gamma[2], - gamma[1])).simplify() 
        + chi_wp_muj.lhs.subs(j,2))
    ),
    Eq(
        8*H_b_s[2].lhs, 
        8*((H_b_s[2].rhs - chi_wp_z1.rhs).simplify() 
        + chi_wp_z1.lhs.subs(j,2))
    )
]
H_b_s_wp_subs = [_.args for _ in H_b_s_wp]

# Derive log diff u,v in wp, wpp

dlog_ubars_wp = [
    Eq(
        _.lhs, 
        _.rhs
        .subs(H_b_s_wp_subs)
        .subs([
            _.subs(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]), w(z,_j+1)).args for _j, _ in enumerate(uvbar_wps)
        ])
        .doit()
        .replace(diff(wp(wild,g2,g3),wild), wpp(wild, g2, g3))
        .doit()
        .collect(chi, factor)
    ) 
    for _ in dlog_ubars
]

# Write in compact form
_s_j_vals = [(s(1), 1), (s(2), 1), (s(3), 2)]
mu3_z0_z1 = Eq(mu[3], z1 + z0)

dlog_ubar_j = Eq(
    diff(ubar(z, mu[j]),z)/ubar(z, mu[j]),
    (wpp(mu[j] - z0, g2, g3) - wpp(z - z0, g2, g3))/(wp(mu[j] - z0, g2, g3) - wp(z - z0, g2, g3))/2
    + chi*gammabar[j]/s(j) + 1/chi
)

for _j, _ in enumerate(dlog_ubars_wp):
    assert (_.lhs - _.rhs).subs([
        dlog_ubar_j.subs(j, _j+1).subs(_s_j_vals).args,
        mu3_z0_z1.args
    ]).simplify() == 0, 'compact j form does not reproduce each u_j'



for _ in H_b_s:
    _
print()
chi_wp_muj
chi_wp_z1
print()
for _ in H_b_s_wp:
    _
print()
for _ in dlog_ubars_wp:
    _
print()
dlog_ubar_j

Eq(Hbar*chi/2 - 11*chi**2*gammabar[1]**2/16 - chi**2*gammabar[1]*gammabar[2]/8 - 3*chi**2*gammabar[2]**2/16 - 2*gammabar[1], -(b[1] + 2*b[2]*lamda[1])*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)/(2*(gamma[1] - lamda[1])) - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*b[2]/2 + 2 + 4*lamda[1]/(gamma[1] - lamda[1]))

Eq(Hbar*chi/2 - 3*chi**2*gammabar[1]**2/16 - chi**2*gammabar[1]*gammabar[2]/8 - 11*chi**2*gammabar[2]**2/16 - 2*gammabar[2], (b[1] + 2*b[2]*lamda[1])*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)/(2*(gamma[1] + lamda[1])) - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*b[2]/2 + 2 - 4*lamda[1]/(gamma[1] + lamda[1]))

Eq(Hbar*chi - chi**2*gammabar[1]**2/2 - chi**2*gammabar[1]*gammabar[3]/2 - 5*chi**2*gammabar[3]**2/8 - 2*gammabar[3], -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*b[2])

Eq(chi*\wp'(-z0 + mu[j], g2, g3)/4, -(b[1] + 2*b[2]*lamda[1])*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)/(2*gamma[j] - 2*lamda[1]) - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*b[2]/2 + 2 + 4*lamda[1]/(gamma[j] - lamda[1]))

Eq(chi*\wp'(z1, g2, g3)/2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*b[2])

...

Eq(8*Hbar*chi - 4*chi**2*gammabar[1]**2 - 4*chi**2*gammabar[1]*gammabar[3] - 5*chi**2*gammabar[3]**2 - 16*gammabar[3], 4*chi*\wp'(z1, g2, g3))

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), (-\wp'(z - z0, g2, g3) + \wp'(-z0 + mu[1], g2, g3))/(2*(-wp(z - z0, g2, g3) + wp(-z0 + mu[1], g2, g3))) + (chi**2*gammabar[1] + 1)/chi)

Eq(Derivative(ubar(z, mu[2]), z)/ubar(z, mu[2]), (-\wp'(z - z0, g2, g3) + \wp'(-z0 + mu[2], g2, g3))/(2*(-wp(z - z0, g2, g3) + wp(-z0 + mu[2], g2, g3))) + (chi**2*gammabar[2] + 1)/chi)

Eq(Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]), (-\wp'(z1, g2, g3) + \wp'(z - z0, g2, g3))/(2*(-wp(z1, g2, g3) + wp(z - z0, g2, g3))) + (chi**2*gammabar[3] + 2)/(2*chi))

Eq(Derivative(ubar(z, mu[j]), z)/ubar(z, mu[j]), chi*gammabar[j]/s(j) + (-\wp'(z - z0, g2, g3) + \wp'(-z0 + mu[j], g2, g3))/(2*(-wp(z - z0, g2, g3) + wp(-z0 + mu[j], g2, g3))) + 1/chi)

We then use the Weierstrass elliptic function hierarchy to solve. The below solutions have been numerically verified in the numeric plots notebook:

In [23]:
dlog_ubar_j_zeta = dlog_ubar_j.subs([pw_to_zw_identity.subs([(z, mu[j] - z0), (x, z-z0)]).args])
dlog_ubar_j_log_sig = dlog_ubar_j_zeta.subs([
    zw_dlog_sigma_identity.subs([(x, z0)]).args,
    zw_dlog_sigma_identity.subs([(x, 2*z0 - mu[j])]).args
])
ubar_sig_sol = Eq(
    exp(Integral(dlog_ubar_j_log_sig.lhs, z).doit()), 
    alphabar[j]*exp(Integral(dlog_ubar_j_log_sig.rhs, z).doit()).simplify()
)

uvbar_j_wp = Eq(
    ubar(z, mu[j])*vbar(z, mu[j]), 
    s(j)*(wp(mu[j] - z0, g2, g3) - wp(z - z0, g2, g3))/2
)
for _j, _ in enumerate(uvbar_wps):
    assert (_.lhs - _.rhs).subs([
        uvbar_j_wp.subs(j, _j+1).subs(_s_j_vals).args,
        mu3_z0_z1.args
    ]).simplify() == 0, 'compact j form does not reproduce each u_j*v_j'
    
uvbar_j_sig = uvbar_j_wp.subs([sigma_p_identity.subs([(y, mu[j] - z0), (x, z-z0)]).args])
vbar_sig_sol = Eq(vbar(z, mu[j]), solve(uvbar_j_sig.subs([ubar_sig_sol.args]), vbar(z, mu[j]))[0])


dlog_ubar_j_zeta
dlog_ubar_j_log_sig
ubar_sig_sol
vbar_sig_sol
uvbar_j_wp

Eq(Derivative(ubar(z, mu[j]), z)/ubar(z, mu[j]), chi*gammabar[j]/s(j) - zeta(z - z0, g2, g3) - zeta(-z0 + mu[j], g2, g3) + zeta(z - 2*z0 + mu[j], g2, g3) + 1/chi)

Eq(Derivative(ubar(z, mu[j]), z)/ubar(z, mu[j]), chi*gammabar[j]/s(j) - zeta(-z0 + mu[j], g2, g3) - Derivative(log(sigma(z - z0, g2, g3)), z) + Derivative(log(sigma(z - 2*z0 + mu[j], g2, g3)), z) + 1/chi)

Eq(ubar(z, mu[j]), sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*(chi*gammabar[j]/s(j) - zeta(-z0 + mu[j], g2, g3) + 1/chi))*alphabar[j]/sigma(z - z0, g2, g3))

Eq(vbar(z, mu[j]), s(j)*sigma(z - mu[j], g2, g3)*exp(z*(-chi*gammabar[j]/s(j) + zeta(-z0 + mu[j], g2, g3) - 1/chi))/(2*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)**2*alphabar[j]))

Eq(ubar(z, mu[j])*vbar(z, mu[j]), (-wp(z - z0, g2, g3) + wp(-z0 + mu[j], g2, g3))*s(j)/2)

In [24]:
chi_gam3_b_lam = Eq(
    chi*gammabar[3]/2 - zeta(z1, g2, g3) + 1/chi,
    (chi*gammabar[3]/2 - zeta(z1, g2, g3) + 1/chi)
    .subs(gammabar[3], -gammabar[1] - gammabar[2])
    .subs(_gam_lam_subs)
    .subs(params_to_bj_lam)
    .factor()
    .subs([gamma_lamda_bj_gam1.args])
    .subs(b[0], x- b[1]*lamda[1]- b[2]*lamda[1]**2)
    .expand().collect(x, factor)
    .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
    .expand()
)

ubar3_b = Eq(
    ubar_sig_sol.lhs
    .subs(j,3), 
    ubar_sig_sol.rhs
    .subs([(j,3), mu3_z0_z1.args])
    .subs(_s_j_vals)
    .subs([chi_gam3_b_lam.args])
)
vbar3_b = Eq(
    vbar_sig_sol.lhs
    .subs(j,3), 
    vbar_sig_sol.rhs
    .subs([(j,3), mu3_z0_z1.args])
    .subs(_s_j_vals)
    .subs([chi_gam3_b_lam.args])
)

ubar3_b
vbar3_b

Eq(ubar(z, mu[3]), sigma(z - z0 + z1, g2, g3)*exp(z*(-zeta(z1, g2, g3) + b[1]/4 + b[2]*lamda[1]/2 + lamda[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)))*alphabar[3]/sigma(z - z0, g2, g3))

Eq(vbar(z, mu[3]), sigma(z - z0 - z1, g2, g3)*exp(z*(zeta(z1, g2, g3) - b[1]/4 - b[2]*lamda[1]/2 - lamda[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)))/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)*alphabar[3]))

### Check theta

In [38]:
uhatz_sol = Eq(
    uhat(z, mu[j]),
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)
    *exp(-z*(zeta(-z0 + z1 + mu[j], g2, g3) + b[2]*gamma[j]))*alphahat[j]/
    (
        sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))
        *sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)*sqrt(b[2])
    )
)
vhatz_sol = Eq(
    vhat(z, mu[j]),
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[j], g2, g3)
    *exp(z*(zeta(-z0 + z1 + mu[j], g2, g3) + b[2]*gamma[j]))/
    (
        sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))
        *sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 + z1, g2, g3)*alphahat[j]*sqrt(b[2])
    )
)

uhatz_sol
vhatz_sol

Eq(uhat(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)*exp(-z*(zeta(-z0 + z1 + mu[j], g2, g3) + b[2]*gamma[j]))*alphahat[j]/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)*sqrt(b[2])))

Eq(vhat(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[j], g2, g3)*exp(z*(zeta(-z0 + z1 + mu[j], g2, g3) + b[2]*gamma[j]))/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 + z1, g2, g3)*alphahat[j]*sqrt(b[2])))

In [109]:
theta_b_gam_lam = Eq(theta, -b[2]*gamma[1] + 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))
_check_theta = Eq(
    2*I*pi*m,
    (
        zeta(-z0 + z1 + mu[1], g2, g3) + b[2]*gamma[1] + 
        (-b[2]*gamma[1] + 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)) +
        (chi*(chi*gammabar[1] - zeta(-z0 + mu[1], g2, g3)) + 1)/chi + 
        (chi*(chi*gammabar[3] - 2*zeta(z1, g2, g3)) + 2)/(2*chi)
    ).subs([
        (gammabar[3], solve(gammabar_2_as_1.subs(gammabar[2], - gammabar[1] - gammabar[3]), gammabar[3])[0]),
        gammabar_1_2_as_gamma[0].args,
        d5_bj_lam.args,
        chi_bj_lam.args,
        gamma_lamda_bj_gam1.args
    ]).simplify()
    .subs(b[0], x- b[1]*lamda[1] - b[2]*lamda[1]**2)
    .expand().collect([
        zeta(z1, g2, g3),
        zeta(-z0+mu[1], g2, g3),
        zeta(z1-z0+mu[1], g2, g3),
        x
    ],factor)
    .subs(x, b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)
    .subs([
        (pw_to_zw_identity.subs([(x,z1),(z, mu[1]-z0)]).rhs, pw_to_zw_identity.subs([(x,z1),(z, mu[1]-z0)]).lhs)
    ])
)

_wp_b_gam_term = Eq(
    ((C_wpz1_z0_muj_to_d_lam1_gamj.lhs - B_wpz1_z0_muj_to_d_lam1_gamj.lhs)/2)
    .simplify(),
    (C_wpz1_z0_muj_to_d_lam1_gamj.rhs - B_wpz1_z0_muj_to_d_lam1_gamj.rhs)/2
)


assert (
    (_check_theta.rhs + _wp_b_gam_term.subs(j,1).rhs - _wp_b_gam_term.subs(j,1).lhs).factor()
).subs([gamma_lamda_bj_gam1.args]).simplify() == 0, 'theta check failed'